# Challenge Three: Developing Multi-Agent Systems

**Goal:** Demonstrate the ability to program multi-agent systems using the Google
Agent Development Kit (ADK).

This notebook is a copy of the Challenge Two notebook, restructured into a
three-agent system:
1. **`weather_agent`** -- specialist sub-agent that answers US weather questions
   (same tools and validation/logging callbacks as Challenges One and Two).
2. **`search_agent`** -- specialist sub-agent that answers general/current-events
   questions using the ADK's built-in Google Search tool.
3. **`root_agent`** -- coordinating agent that receives the user's request and
   delegates it to whichever specialist sub-agent fits.




In [1]:
# 1. Install dependencies
!pip install --upgrade --quiet google-adk google-cloud-aiplatform litellm requests openai


In [2]:
# 2. Imports and configuration
import os
import re
import logging
import requests
from typing import Optional, List, Dict, Tuple

from google.adk.agents import Agent
from google.adk.tools import google_search
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse

# --- Configuration ---
# Set these as environment variables (recommended) or fill in directly.
# GOOGLE_MAPS_API_KEY: Google Maps Platform API key with the Geocoding API enabled.
# GOOGLE_API_KEY / GOOGLE_CLOUD_PROJECT: used by ADK/Vertex for the Gemini model.

GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "YOUR_GOOGLE_MAPS_API_KEY")

MODEL_GEMINI_FLASH = "gemini-2.5-flash"

# --- Logging setup for the callback examples below ---
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("weather_agent_callbacks")


## Weather agent: tools, validation helpers, and callbacks

Unchanged from Challenge Two -- the weather specialist keeps its own tools,
input validation, and logging callbacks. Note the built-in `google_search`
tool used by the search agent below cannot be mixed with other tools on the
*same* agent, which is one reason this stays a dedicated sub-agent.


In [3]:
# 3. Tool: convert a place name to latitude/longitude using the Google Maps Geocoding API
def get_lat_lon(place: str) -> Optional[Tuple[float, float]]:
    """
    Convert a place name (e.g. a city and state) into geographic coordinates
    using the Google Maps Geocoding API.

    Args:
        place (str): A human-readable location, e.g. "Chicago, IL" or
            "1600 Amphitheatre Parkway, Mountain View, CA".

    Returns:
        Optional[Tuple[float, float]]: A (latitude, longitude) tuple, or
        None if the location could not be geocoded or an error occurred.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": place, "key": GOOGLE_MAPS_API_KEY}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data.get("status") != "OK" or not data.get("results"):
            return None

        location = data["results"][0]["geometry"]["location"]
        return (location["lat"], location["lng"])
    except (requests.RequestException, KeyError, IndexError):
        return None


In [4]:
# 4. Tool: fetch the extended weather forecast from the National Weather Service API
def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """
    Fetch the extended weather forecast from the U.S. National Weather Service API
    based on a given latitude and longitude.

    Args:
        lat (float): Latitude of the location (e.g., 38.8977).
        lon (float): Longitude of the location (e.g., -77.0365).

    Returns:
        Optional[List[Dict[str, str]]]: A list of forecast period dictionaries,
        each containing keys such as "name", "temperature", "temperatureUnit",
        "windSpeed", "windDirection", "shortForecast", and "detailedForecast".
        Returns None if data is unavailable or an error occurs.
    """
    headers = {"User-Agent": "adk-weather-alerts-agent (contact: akhil.sharma@wwt.com)"}

    try:
        # Step 1: resolve the lat/lon to a NWS gridpoint / forecast URL.
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        points_resp = requests.get(points_url, headers=headers, timeout=10)
        points_resp.raise_for_status()
        forecast_url = points_resp.json()["properties"]["forecast"]

        # Step 2: fetch the extended forecast periods.
        forecast_resp = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_resp.raise_for_status()
        periods = forecast_resp.json()["properties"]["periods"]

        return [
            {
                "name": p.get("name", ""),
                "temperature": str(p.get("temperature", "")),
                "temperatureUnit": p.get("temperatureUnit", ""),
                "windSpeed": p.get("windSpeed", ""),
                "windDirection": p.get("windDirection", ""),
                "shortForecast": p.get("shortForecast", ""),
                "detailedForecast": p.get("detailedForecast", ""),
            }
            for p in periods
        ]
    except (requests.RequestException, KeyError, IndexError):
        return None


In [5]:
# 5. Moderation: lightweight check for malicious / policy-violating input
def check_user_input(user_text: str) -> str:
    """
    Very lightweight moderation check for the user's raw message text.

    Args:
        user_text (str): The raw user message.

    Returns:
        str: "BAD" if the input looks malicious or off-policy, otherwise "OK".
    """
    lowered = user_text.lower()
    suspicious_patterns = [
        "ignore previous instructions",
        "ignore all previous instructions",
        "disregard your instructions",
        "disregard all prior instructions",
        "reveal your system prompt",
        "reveal your instructions",
        "you are now",
        "jailbreak",
        "<script",
        "drop table",
        "rm -rf",
    ]
    for pattern in suspicious_patterns:
        if pattern in lowered:
            return "BAD"
    return "OK"


In [6]:
# 6. Location validation: confirm a mentioned location is in the United States
LOCATION_PATTERN = re.compile(r"\bin\s+([A-Za-z][A-Za-z .,\'-]*)", re.IGNORECASE)


def extract_location_candidate(user_text: str) -> Optional[str]:
    """
    Pull a best-effort location phrase out of a user message, e.g. extract
    "Denver, CO" from "What's the weather like in Denver, CO?".

    Args:
        user_text (str): The raw user message.

    Returns:
        Optional[str]: The extracted location phrase, or None if no
        "in <location>" pattern was found.
    """
    match = LOCATION_PATTERN.search(user_text)
    if not match:
        return None
    candidate = match.group(1).strip().rstrip("?.!")
    return candidate or None


def get_location_country(place: str) -> Optional[str]:
    """
    Resolve a place name to its ISO 3166-1 alpha-2 country code using the
    Google Maps Geocoding API.

    Args:
        place (str): A human-readable location.

    Returns:
        Optional[str]: A two-letter country code (e.g. "US", "FR"), or None
        if the location could not be resolved.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": place, "key": GOOGLE_MAPS_API_KEY}

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data.get("status") != "OK" or not data.get("results"):
            return None

        for component in data["results"][0].get("address_components", []):
            if "country" in component.get("types", []):
                return component.get("short_name")
        return None
    except (requests.RequestException, KeyError, IndexError):
        return None


def check_location_is_us(user_text: str) -> Optional[str]:
    """
    If the user's message mentions a location, confirm it resolves to the
    United States (the National Weather Service API only covers the US).

    Args:
        user_text (str): The raw user message.

    Returns:
        Optional[str]: None if the check passes, otherwise a rejection message.
    """
    candidate = extract_location_candidate(user_text)
    if not candidate:
        return None

    country = get_location_country(candidate)
    if country is None:
        return None

    if country != "US":
        return (
            f"Sorry, I can only provide weather alerts for locations in the "
            f"United States. \'{candidate}\' appears to be in {country}."
        )
    return None


In [7]:
# 7. After-model callback: log model responses
def log_model_response(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:
    """
    Log the model's response after it comes back, before it is returned to the user.

    Args:
        callback_context (CallbackContext): Metadata about the current agent invocation.
        llm_response (LlmResponse): The response returned by the model.

    Returns:
        Optional[LlmResponse]: Always None -- logging never modifies the response.
    """
    if llm_response.content and llm_response.content.parts:
        txt = llm_response.content.parts[0].text
        if txt:
            logger.info("[%s] MODEL \u00bb %s", callback_context.agent_name, txt.strip())
    return None


In [8]:
# 8. Before-model callback: validate + log (chained)
def moderate_and_validate_user_prompt(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
    """
    Chained before-model callback: reject malicious input or non-US locations,
    otherwise log the prompt and let the agent proceed.

    Args:
        callback_context (CallbackContext): Metadata about the current agent invocation.
        llm_request (LlmRequest): The request about to be sent to the model.

    Returns:
        Optional[LlmResponse]: A response that short-circuits the model call
        if validation fails, otherwise None.
    """
    if not llm_request.contents:
        return None

    last = llm_request.contents[-1]
    if last.role != "user" or not last.parts or not last.parts[0].text:
        return None

    user_text = last.parts[0].text.strip()

    if check_user_input(user_text) == "BAD":
        logger.warning("[%s] BLOCKED (moderation) \u00bb %s", callback_context.agent_name, user_text)
        return LlmResponse(content={
            "role": "model",
            "parts": [{"text": "Sorry, I can't help with that request \u2014 it violates our content guidelines."}],
        })

    rejection = check_location_is_us(user_text)
    if rejection:
        logger.warning("[%s] BLOCKED (non-US location) \u00bb %s", callback_context.agent_name, user_text)
        return LlmResponse(content={"role": "model", "parts": [{"text": rejection}]})

    logger.info("[%s] USER  \u00bb %s", callback_context.agent_name, user_text)
    return None


## Building the three agents

`weather_agent` and `search_agent` are specialists; `root_agent` has no tools
of its own and instead lists both specialists in `sub_agents`, delegating
based on their `name`/`description` and its own routing instructions.


In [9]:
# 9. Weather agent instructions and definition
WEATHER_AGENT_INSTRUCTIONS = """
You are a friendly and knowledgeable weather alerts assistant for locations in the
United States.

When asked about the weather for a place:
1. Use the `get_lat_lon` tool to convert the place name into latitude/longitude.
   If it fails, say you could not find that location and ask for clarification
   (e.g. add a state).
2. Use the `get_extended_weather_forecast` tool with those coordinates to retrieve the
   forecast periods.
3. Summarize the current/upcoming conditions in plain language: temperature, wind, and
   general outlook.
4. Proactively call out anything alert-worthy: extreme heat (>= 95F) or cold (<= 20F),
   high winds (>= 25 mph), or forecasts mentioning storms, tornadoes, snow, or ice. If
   nothing stands out, say conditions look normal.
5. Keep responses concise and easy to scan, and always name the city/location you are
   reporting on.
"""

weather_agent = Agent(
    name="weather_agent",
    model=MODEL_GEMINI_FLASH,
    description=(
        "Specialist agent that answers questions about current weather conditions, "
        "forecasts, and weather alerts for locations in the United States."
    ),
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_extended_weather_forecast, get_lat_lon],
    before_model_callback=moderate_and_validate_user_prompt,
    after_model_callback=log_model_response,
)


In [10]:
# 10. Search agent: uses the ADK built-in Google Search tool
SEARCH_AGENT_INSTRUCTIONS = """
You answer general knowledge and current-events questions using Google Search.
Search for up-to-date information and summarize it concisely and factually.
Do not answer weather-related questions -- those are handled by a different agent.
"""

search_agent = Agent(
    name="search_agent",
    model=MODEL_GEMINI_FLASH,
    description=(
        "Specialist agent that answers general knowledge and current-events "
        "questions by searching the web with Google Search."
    ),
    instruction=SEARCH_AGENT_INSTRUCTIONS,
    tools=[google_search],
    # The built-in google_search tool cannot be combined with any other
    # function-declaration tool in the same model call. By default ADK adds
    # an automatic transfer_to_agent tool to sub-agents so they can hand off
    # to siblings/parent; that collides with google_search and Gemini
    # rejects the request. This agent is a leaf specialist, so disable both.
    disallow_transfer_to_parent=True,
    disallow_transfer_to_peers=True,
)


In [11]:
# 11. Root agent: coordinates and delegates to the two specialists
ROOT_AGENT_INSTRUCTIONS = """
You are a coordinating agent for a small assistant team. You have two
specialists you can delegate to:

- `weather_agent`: use this for any question about current weather conditions,
  forecasts, or weather alerts for a US location.
- `search_agent`: use this for any question that needs up-to-date general
  information from the web (news, facts, "what is / who is", anything not
  about the weather).

Read the user's request, decide which specialist fits best, and delegate to
them. Do not try to answer weather or search questions yourself -- always
hand off to the appropriate specialist. If a request genuinely needs both,
delegate to one, then the other, and combine their answers.
"""

root_agent = Agent(
    name="root_agent",
    model=MODEL_GEMINI_FLASH,
    description="Coordinating agent that routes requests to the weather or search specialist.",
    instruction=ROOT_AGENT_INSTRUCTIONS,
    sub_agents=[weather_agent, search_agent],
)


## Running the root agent

`ask_agent_verbose` prints every streamed event as it arrives -- including
each event's `author` -- so the notebook output makes it clear which
sub-agent actually handled each request, not just the final answer.


In [12]:
# 12. Helper to run a query against an agent and print every event (proves sub-agent delegation)
from vertexai.preview import reasoning_engines
from IPython.display import Markdown, display

def describe_event(event: dict) -> None:
    """
    Print a one-line-per-part summary of a single streamed event: which agent
    authored it, and whether it is text, a tool call, or a tool result.

    Args:
        event (dict): One event dict from AdkApp.stream_query().
    """
    author = event.get("author", "?")
    content = event.get("content") or {}
    for part in content.get("parts", []) or []:
        if part.get("text"):
            text = part.get("text", "").strip()[:200]
            print(f"  [{author}] TEXT  \u00bb {text}")
        elif part.get("function_call"):
            fc = part["function_call"]
            fc_name = fc.get("name")
            fc_args = fc.get("args")
            print(f"  [{author}] CALL  \u00bb {fc_name}({fc_args})")
        elif part.get("function_response"):
            fr = part["function_response"]
            fr_name = fr.get("name")
            print(f"  [{author}] RESULT \u00bb from {fr_name}")


def ask_agent_verbose(agent: Agent, question: str, user_id: str = "test-user-id") -> Optional[str]:
    """
    Host the given agent in an AdkApp, create a session, query it once, and
    print every event along the way (including which sub-agent handled it)
    before returning the final response text.

    Args:
        agent (Agent): The ADK agent to run (typically the root agent).
        question (str): The natural-language question/prompt to send.
        user_id (str): An identifier for the querying user/session owner.

    Returns:
        Optional[str]: The text of the agent\'s final response, or None on error.
    """
    app = reasoning_engines.AdkApp(agent=agent)
    session = app.create_session(user_id=user_id)
    session_id = session["id"] if isinstance(session, dict) else session.id

    last_event = None
    try:
        for event in app.stream_query(user_id=user_id, session_id=session_id, message=question):
            describe_event(event)
            last_event = event
    except Exception as e:
        print(f"Error while querying agent \'{agent.name}\': {e}")
        return None

    if not last_event or "content" not in last_event:
        print(f"Agent \'{agent.name}\' did not return a valid final response.")
        return None

    return last_event["content"]["parts"][0]["text"]


## Test code

Two requests that should each be routed to a different sub-agent, plus the
event trace printed above each answer showing `weather_agent` or
`search_agent` (not `root_agent`) doing the actual work.


In [13]:
# 13. Test 1: a weather question -- should be delegated to weather_agent
print("=== Query: weather question ===")
response = ask_agent_verbose(root_agent, "What's the weather like in Austin, TX? Any alerts I should know about?")
print()
display(Markdown(response or "*(no response)*"))


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


=== Query: weather question ===


/usr/local/lib/python3.12/dist-packages/google/adk/tools/transfer_to_agent_tool.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  function_decl = super()._get_declaration()


  [root_agent] CALL  » transfer_to_agent({'agent_name': 'weather_agent'})
  [root_agent] RESULT » from transfer_to_agent
  [weather_agent] CALL  » get_lat_lon({'place': 'Austin, TX'})
  [weather_agent] RESULT » from get_lat_lon
  [weather_agent] CALL  » get_extended_weather_forecast({'lat': 30.267153, 'lon': -97.7430608})
  [weather_agent] RESULT » from get_extended_weather_forecast
  [weather_agent] TEXT  » Here's the weather for Austin, TX:

**Today:** Sunny with a high near 104°F and a heat index as high as 107°F. Winds will be light from the south southwest, 0 to 5 mph.
**Tonight:** Partly cloudy with



Here's the weather for Austin, TX:

**Today:** Sunny with a high near 104°F and a heat index as high as 107°F. Winds will be light from the south southwest, 0 to 5 mph.
**Tonight:** Partly cloudy with a low around 80°F and a heat index as high as 106°F. Winds will be light from the south southwest, 0 to 5 mph.

**Alerts:**
*   **Extreme Heat:** Temperatures will be very high, reaching 104°F today and staying around 100°F or above for the next several days. Heat index values will be as high as 107°F today and 108°F on Thursday. Take precautions against the heat.
*   There's a **30% chance of showers and thunderstorms** on Thursday and Thursday night.

In [14]:
# 14. Test 2: a general-knowledge question -- should be delegated to search_agent
print("=== Query: general knowledge question ===")
response = ask_agent_verbose(root_agent, "Who won the most recent Super Bowl?")
print()
display(Markdown(response or "*(no response)*"))


=== Query: general knowledge question ===


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:975: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


  [root_agent] CALL  » transfer_to_agent({'agent_name': 'search_agent'})
  [root_agent] RESULT » from transfer_to_agent
  [search_agent] TEXT  » The most recent Super Bowl was won by the Seattle Seahawks in Super Bowl LX (60) in 2026, where they defeated the New England Patriots with a score of 29-13.



The most recent Super Bowl was won by the Seattle Seahawks in Super Bowl LX (60) in 2026, where they defeated the New England Patriots with a score of 29-13.

## Notes

- Replace the placeholder `GOOGLE_MAPS_API_KEY` in the configuration cell with a real
  value before running.
- The `google_search` built-in tool cannot be combined with other tools on the same
  agent -- that's why `search_agent` only has `google_search` and the weather tools
  live on a separate `weather_agent`.
- Watch the `[weather_agent] ...` / `[search_agent] ...` lines in the event trace for
  each test: they confirm the root agent actually delegated to the right specialist,
  rather than answering directly itself.
